# TD – Construction de la collection entreprise

Nous partons de l'export KBO Open Data : huit fichiers CSV montés sur un volume Docker et une instance MongoDB vierge. Ce TD a pour objectif de bâtir, étape par étape, un pipeline d'ingestion qui importe ces fichiers dans MongoDB, puis les fusionne pour constituer une collection entreprise. Chaque document résultant représentera une entreprise et intégrera ses établissements ainsi que ses succursales sous forme de sous-documents imbriqués.

# 0) Objectif du TD
**Le modèle de données KBO**

La KBO (Kruispuntbank van Ondernemingen) est le registre central des entreprises en Belgique. Elle publie ses données sous forme de fichiers CSV distincts, à l'image d'un export de base de données relationnelle.

Les données s'organisent autour de trois types d'entités :

Niveau	Fichier	Clé principale	Description
Entreprise	enterprise.csv	EnterpriseNumber	Personne juridique
Établissement	establishment.csv	EstablishmentNumber	Unité opérationnelle belge (magasin, siège, usine, etc.)
Succursale	branch.csv	Id	Présence belge d'une entreprise étrangère

Quatre fichiers additionnels enrichissent ces entités avec des informations complémentaires :

denomination.csv
address.csv
contact.csv
activity.csv

Tous partagent la même clé de rattachement : EntityNumber. Cette colonne peut référencer aussi bien une entreprise, qu'un établissement ou une succursale, selon le contexte.

C'est sur ce point que repose l'essentiel du TD. Les fichiers de détails n'indiquent pas à quel type d'entité ils appartiennent : ils ne fournissent qu'un identifiant. La même logique de jointure peut donc s'appliquer aux trois niveaux, ce qui rend possible l'écriture d'une fonction réutilisable.

**Du modèle relationnel au modèle documentaire**

Dans le modèle d'origine, accéder à l'ensemble des informations d'une entreprise impose de parcourir jusqu'à huit fichiers distincts.

La couche Bronze a pour vocation d'effectuer ce travail une seule fois, lors d'un traitement batch, afin de produire un document MongoDB autonome qui regroupe l'intégralité des informations d'une entreprise.

---

**La couche Bronze**

La couche Bronze reproduit à l'identique les données sources, sans les altérer.

Aucune logique métier n'est appliquée :

les codes sont conservés bruts (Status = "AC", TypeOfAddress = "REGO"),
les colonnes multilingues sont toutes préservées (MunicipalityNL, MunicipalityFR),
les valeurs manquantes ne sont pas traitées.

Le seul rôle de cette couche est de consolider les informations pour former des documents complets. Cette démarche garantit une couche facilement reproductible et strictement fidèle à la source. L'interprétation et le nettoyage seront pris en charge dans la couche Silver.

**Configuration**

Le notebook repose sur trois paramètres de configuration, chacun lu depuis une variable d'environnement avec une valeur par défaut. Ce mécanisme permet d'exécuter le même notebook sans aucun ajustement, que ce soit dans le conteneur Docker du TD (/data/kbo, mongodb://mongo:27017) ou sur un poste local.

In [1]:
import csv
import os
import time
from itertools import islice
from pathlib import Path

import pymongo

def resolve_data_dir() -> Path:
    """Retourne le premier répertoire candidat qui contient effectivement l'export KBO."""
    candidates = (os.getenv("KBO_DATA_DIR"), "/data/kbo", Path.cwd(), Path.cwd().parent)
    for candidate in candidates:
        if candidate and Path(candidate).joinpath("meta.csv").exists():
            return Path(candidate)
    raise FileNotFoundError(
        "Export KBO introuvable. Définissez KBO_DATA_DIR sur le dossier "
        f"contenant meta.csv (candidats testés : {candidates})."
    )

DATA_DIR = resolve_data_dir()
MONGO_URI = os.getenv("MONGO_URI", "mongodb://localhost:27017")
DB_NAME = os.getenv("MONGO_DB", "kbo")

client = pymongo.MongoClient(MONGO_URI, socketTimeoutMS=None)
db = client[DB_NAME]

print("donnees :", DATA_DIR)
print("mongodb :", MONGO_URI, "->", DB_NAME)
print("serveur :", client.server_info()["version"])

donnees : /home/jovyan/work
mongodb : mongodb://mongo:27017 -> kbo
serveur : 8.2.12


Aperçu de la volumétrie avant de démarrer : ces chiffres conditionnent l'ensemble
des décisions techniques qui suivent.

In [2]:
CSV_FILES = {
    # fichier               collection cible        identifiant (_id)
    "meta.csv":          ("kbo_meta",          None),
    "code.csv":          ("kbo_code",          None),
    "enterprise.csv":    ("kbo_enterprise",    "EnterpriseNumber"),
    "establishment.csv": ("kbo_establishment", None),
    "branch.csv":        ("kbo_branch",        None),
    "denomination.csv":  ("kbo_denomination",  None),
    "address.csv":       ("kbo_address",       None),
    "contact.csv":       ("kbo_contact",       None),
    "activity.csv":      ("kbo_activity",      None),
}

print(f"{'fichier':<20}{'taille':>10}   colonnes")
for filename in CSV_FILES:
    path = DATA_DIR / filename
    with path.open(encoding="utf-8", newline="") as handle:
        header = next(csv.reader(handle))
    print(f"{filename:<20}{path.stat().st_size / 1e6:>9,.0f}M   {', '.join(header)}")

fichier                 taille   colonnes
meta.csv                    0M   Variable, Value
code.csv                    2M   Category, Code, Language, Description
enterprise.csv             90M   EnterpriseNumber, Status, JuridicalSituation, TypeOfEnterprise, JuridicalForm, JuridicalFormCAC, StartDate
establishment.csv          71M   EstablishmentNumber, StartDate, EnterpriseNumber
branch.csv                  0M   Id, StartDate, EnterpriseNumber
denomination.csv          155M   EntityNumber, Language, TypeOfDenomination, Denomination
address.csv               305M   EntityNumber, TypeOfAddress, CountryNL, CountryFR, Zipcode, MunicipalityNL, MunicipalityFR, StreetNL, StreetFR, HouseNumber, Box, ExtraAddressInfo, DateStrikingOff
contact.csv                35M   EntityNumber, EntityContact, ContactType, Value
activity.csv            1,528M   EntityNumber, ActivityGroup, NaceVersion, NaceCode, Classification


# 1) Import des fichiers CSV dans MongoDB

Les fichiers de l'export KBO peuvent atteindre des tailles considérables. L'objectif de cette étape est de concevoir une fonction capable d'importer un fichier CSV dans une collection MongoDB par lots successifs, puis de l'appliquer sur les huit fichiers de l'export, en dédiant une collection à chacun.

**Principe du chargement**

Certains fichiers, comme activity.csv, comptent plusieurs dizaines de millions de lignes. Il est donc hors de question de les charger intégralement en mémoire.

Pour un import performant, on s'appuie sur trois mécanismes :

csv.DictReader parcourt le fichier ligne à ligne, sans le matérialiser.
iter_batches regroupe les lignes en lots pour maintenir une empreinte mémoire réduite.
insert_many(ordered=False) envoie chaque lot en une seule requête, ce qui maximise le débit.

**Bonnes pratiques**

Quelques réglages garantissent un import fiable :

utiliser encoding="utf-8" pour préserver les caractères accentués ;
utiliser newline="" afin de déléguer la gestion des fins de ligne au module csv ;
désigner EnterpriseNumber comme clé _id pour enterprise.csv, ce qui prévient les doublons et rend l'import idempotent.

**Reprise sur erreur**

La fonction doit également gérer la reprise d'un traitement interrompu :

si la collection est déjà entièrement chargée, elle est laissée intacte ;
si elle n'est que partiellement remplie, elle est supprimée puis rechargée depuis zéro.

Ce comportement permet de relancer le pipeline sans avoir à recommencer l'ensemble de l'import.

In [3]:
BATCH_SIZE = 50_000

def iter_batches(iterable, size: int):
    """Découpe un itérable en listes de `size` éléments sans le charger entièrement en mémoire."""
    iterator = iter(iterable)
    while batch := list(islice(iterator, size)):
        yield batch


def count_csv_rows(csv_path: Path) -> int:
    """Compte les lignes de données d'un fichier CSV (hors en-tête), par lecture en streaming."""
    with csv_path.open("r", encoding="utf-8", newline="") as handle:
        return sum(1 for _ in handle) - 1


def load_csv_to_collection(csv_path: Path, collection_name: str, *,
                           id_field: str | None = None,
                           batch_size: int = BATCH_SIZE,
                           skip_if_complete: bool = True) -> int:
    """Importe un fichier CSV dans une collection MongoDB par lots, à empreinte mémoire constante."""
    collection = db[collection_name]
    expected = count_csv_rows(csv_path)

    if skip_if_complete and collection.estimated_document_count() == expected:
        print(f"-- {collection_name:<20} déjà complet ({expected:>12,}) - ignoré")
        return expected

    collection.drop()                      # rechargement complet = snapshot frais
    inserted, started = 0, time.perf_counter()

    with csv_path.open("r", encoding="utf-8", newline="") as handle:
        for batch in iter_batches(csv.DictReader(handle), batch_size):
            if id_field:                   # clé naturelle promue en _id
                for document in batch:
                    document["_id"] = document[id_field]
            collection.insert_many(batch, ordered=False)
            inserted += len(batch)

    elapsed = time.perf_counter() - started
    print(f"OK {collection_name:<20} {inserted:>12,} docs en {elapsed:7.1f}s "
          f"({inserted / elapsed:>9,.0f}/s)")
    return inserted

Application de la fonction aux 9 fichiers de l'export : les 8 fichiers de données
ainsi que `meta.csv`, qui enregistre la date du snapshot.

In [4]:
started = time.perf_counter()
total = sum(
    load_csv_to_collection(DATA_DIR / filename, collection_name, id_field=id_field)
    for filename, (collection_name, id_field) in CSV_FILES.items()
)
print(f"\nTOTAL {total:,} documents en {(time.perf_counter() - started) / 60:.1f} min")

OK kbo_meta                        5 docs en     0.1s (       66/s)


OK kbo_code                   21,468 docs en     0.3s (   64,137/s)


OK kbo_enterprise          1,955,776 docs en    20.9s (   93,471/s)


OK kbo_establishment       1,691,048 docs en    17.6s (   96,252/s)
OK kbo_branch                  7,319 docs en     0.1s (   58,974/s)


OK kbo_denomination        3,354,096 docs en    35.2s (   95,319/s)


OK kbo_address             2,886,648 docs en    54.5s (   52,999/s)


OK kbo_contact               708,661 docs en     7.5s (   94,310/s)


OK kbo_activity           34,373,618 docs en   379.3s (   90,626/s)

TOTAL 44,998,639 documents en 10.8 min


### Vérification

Comparaison entre le nombre de documents chargés et le nombre de lignes des CSV :
contrôle d'intégrité de base à effectuer après toute ingestion.

In [5]:
print(f"{'collection':<20}{'documents':>14}{'lignes CSV':>14}   etat")
for filename, (collection_name, _) in CSV_FILES.items():
    loaded = db[collection_name].count_documents({})
    rows = count_csv_rows(DATA_DIR / filename)
    print(f"{collection_name:<20}{loaded:>14,}{rows:>14,}   "
          f"{'OK' if loaded == rows else 'ECART'}")

print("\nsnapshot :", {d["Variable"]: d["Value"] for d in db.kbo_meta.find()})

collection               documents    lignes CSV   etat
kbo_meta                         5             5   OK


kbo_code                    21,468        21,468   OK


kbo_enterprise           1,955,776     1,955,776   OK


kbo_establishment        1,691,048     1,691,048   OK
kbo_branch                   7,319         7,319   OK


kbo_denomination         3,354,096     3,354,096   OK


kbo_address              2,886,648     2,886,648   OK


kbo_contact                708,661       708,661   OK


kbo_activity            34,373,618    34,373,618   OK

snapshot : {'SnapshotDate': '24-07-2026', 'ExtractTimestamp': '25-07-2026 08:17:43', 'ExtractType': 'full', 'ExtractNumber': '431', 'Version': '1.0.0'}


# 2) Création des index pour les jointures

Pour accélérer les opérations de jointure ($lookup), des index doivent être créés sur les champs servant de clés de liaison entre les collections. Sans eux, MongoDB parcourrait l'intégralité des documents à chaque jointure, rendant le pipeline impraticable en termes de performance.

**Champs à indexer**

Les collections de détails (kbo_denomination, kbo_address, kbo_contact et kbo_activity) doivent être indexées sur EntityNumber, le champ utilisé pour rattacher les informations complémentaires aux entreprises, établissements et succursales.

Les collections kbo_establishment et kbo_branch doivent être indexées sur EnterpriseNumber, afin de relier chaque établissement ou succursale à son entreprise parente.

**Collections sans index supplémentaire**

La collection kbo_enterprise ne demande pas d'index additionnel : en tant que collection principale du pipeline, son identifiant (_id, correspondant à EnterpriseNumber) est automatiquement indexé par MongoDB.

La collection kbo_code n'intervient pas dans cette étape ; elle sera sollicitée lors de la construction de la couche Silver.

**Moment opportun pour créer les index**

Les index sont construits après le chargement des données. Entretenir un index au fil des insertions est bien plus coûteux que de le générer une seule fois sur une collection déjà complète.

In [6]:
JOIN_INDEXES = {
    "kbo_denomination":  "EntityNumber",
    "kbo_address":       "EntityNumber",
    "kbo_contact":       "EntityNumber",
    "kbo_activity":      "EntityNumber",
    "kbo_establishment": "EnterpriseNumber",
    "kbo_branch":        "EnterpriseNumber",
}

for collection_name, field in JOIN_INDEXES.items():
    started = time.perf_counter()
    index_name = db[collection_name].create_index(field)
    print(f"{collection_name:<20} {index_name:<22} {time.perf_counter() - started:>7.1f}s")

kbo_denomination     EntityNumber_1             7.1s


kbo_address          EntityNumber_1             9.0s


kbo_contact          EntityNumber_1             1.2s


kbo_activity         EntityNumber_1            83.0s


kbo_establishment    EnterpriseNumber_1         3.7s
kbo_branch           EnterpriseNumber_1         0.1s


In [7]:
stats = db.command("dbStats", scale=1024 * 1024)
print(f"donnees : {stats['dataSize']:>8,.0f} Mo")
print(f"stockage: {stats['storageSize']:>8,.0f} Mo  (compression zstd)")
print(f"index   : {stats['indexSize']:>8,.0f} Mo")

donnees :    6,719 Mo
stockage:      656 Mo  (compression zstd)
index   :      950 Mo


---

## 3) Rattacher les détails d'une entité

Les trois niveaux d'entités — entreprise, établissement, succursale — nécessitent
chacun les mêmes quatre informations complémentaires : leurs dénominations,
leurs adresses, leurs contacts et leurs activités. Ces informations sont stockées
dans des collections séparées, et s'y rattachent toujours selon la même logique,
quel que soit le type d'entité concerné.

Écrivez une fonction réutilisable qui, à partir du nom du champ à employer
comme clé de jointure côté entité courante, génère les étapes d'agrégation
nécessaires pour récupérer ces quatre informations. Cette fonction sera appelée
trois fois dans la suite du TD — une fois par niveau d'entité — avec à chaque
fois un nom de champ différent.

### L'asymétrie qui rend la fonction réutilisable

Le rattachement d'un détail à son entité est **asymétrique** :

- du **côté du détail**, le champ est *toujours* `EntityNumber` — les quatre
  collections ont été conçues ainsi ;
- du **côté de l'entité**, le champ **change de nom** selon le niveau :
  `EnterpriseNumber`, `EstablishmentNumber`, ou `Id`.

Toute la variabilité se concentre donc dans **un seul paramètre**, `primary_key`.
C'est précisément ce que l'énoncé demande : une unique fonction, trois appels distincts.

La fonction retourne une **liste de 4 étapes** (et non un `$lookup` isolé), ce qui
permet de la décompresser avec `*` à l'emplacement voulu dans le pipeline.

| Appel | `primary_key` | Contexte |
|---|---|---|
| `_detail_lookups("EnterpriseNumber")` | `EnterpriseNumber` | question 6, au niveau entreprise |
| `_detail_lookups("EstablishmentNumber")` | `EstablishmentNumber` | question 5, dans le sous-pipeline établissement |
| `_detail_lookups("Id")` | `Id` | question 4, dans le sous-pipeline succursale |

> **Pourquoi les succursales reçoivent-elles aussi contacts et activités ?**
> En pratique, une succursale n'en possède jamais. Mais appliquer les 4 jointures
> de manière uniforme produit simplement `contacts: []` et `activities: []`.
> Un tableau vide coûte quelques octets ; une exception dans le code coûte
> bien plus cher à maintenir. La couche Silver se chargera de supprimer ces clés définitivement.

In [8]:
DETAIL_SOURCES = (
    ("kbo_denomination", "denominations"),
    ("kbo_address",      "addresses"),
    ("kbo_contact",      "contacts"),
    ("kbo_activity",     "activities"),
)

def _detail_lookups(primary_key: str) -> list[dict]:
    """Génère les 4 étapes d'agrégation qui rattachent les informations complémentaires à une entité.

    `primary_key` désigne le champ identifiant l'entité courante, dont la valeur
    correspond à la colonne `EntityNumber` des 4 collections de détails :
    `EnterpriseNumber`, `EstablishmentNumber` ou `Id` selon le niveau d'entité.
    """
    return [
        {"$lookup": {"from": source,
                     "localField": primary_key,
                     "foreignField": "EntityNumber",
                     "as": alias}}
        for source, alias in DETAIL_SOURCES
    ]


for stage in _detail_lookups("EnterpriseNumber"):
    print(stage)

{'$lookup': {'from': 'kbo_denomination', 'localField': 'EnterpriseNumber', 'foreignField': 'EntityNumber', 'as': 'denominations'}}
{'$lookup': {'from': 'kbo_address', 'localField': 'EnterpriseNumber', 'foreignField': 'EntityNumber', 'as': 'addresses'}}
{'$lookup': {'from': 'kbo_contact', 'localField': 'EnterpriseNumber', 'foreignField': 'EntityNumber', 'as': 'contacts'}}
{'$lookup': {'from': 'kbo_activity', 'localField': 'EnterpriseNumber', 'foreignField': 'EntityNumber', 'as': 'activities'}}


---

## 4) Rattacher les succursales

Une succursale représente la présence en Belgique d'une entreprise étrangère.
Chaque succursale est liée à une entreprise et, comme vu à la question 3,
possède ses propres dénominations et adresses (mais jamais de contacts ni
d'activités).

Écrivez une fonction qui génère l'étape d'agrégation permettant d'associer,
à chaque entreprise, la liste de ses succursales complètes. Chaque succursale
doit déjà embarquer ses propres dénominations et adresses, obtenues via la
fonction de la question 3. Le champ qui lie une entreprise à ses succursales
est différent de celui utilisé à l'intérieur d'une succursale pour récupérer
ses propres détails : les deux corrélations devront donc être rendues explicites.

### Deux clés distinctes dans une seule étape

C'est le point délicat soulevé par l'énoncé. Pour la succursale `9.000.006.626`
de l'entreprise `0257.883.408`, **deux** clés entrent en jeu :

```
entreprise 0257.883.408
   │
   │  (a) lien entreprise -> succursale : kbo_branch.EnterpriseNumber == "0257.883.408"
   ▼
succursale  Id = "9.000.006.626"
   │
   │  (b) lien succursale -> ses détails : kbo_address.EntityNumber == "9.000.006.626"
   ▼
adresse de la succursale
```

La forme abrégée du `$lookup` (`localField` / `foreignField`) ne peut exprimer
qu'**une** corrélation. La forme longue est donc requise :

- **`let`** capture la valeur du document parent — ici `EnterpriseNumber` — et
  l'expose dans une variable `$$enterprise_number` ;
- **`pipeline`** s'exécute *dans le contexte de la collection jointe*
  (`kbo_branch`), où cette variable est accessible ;
- **`$match` + `$expr`** réalise la corrélation (a) : `$expr` est nécessaire
  pour comparer un champ à une variable, ce qu'un `$match` classique ne permet pas ;
- une fois dans ce sous-pipeline, on est « du côté de la succursale » : les
  étapes de la question 3 appliquées à `Id` réalisent la corrélation (b).

> **Le piège** : utiliser `_detail_lookups("EnterpriseNumber")` dans le
> sous-pipeline. On associerait alors à chaque succursale les dénominations et
> adresses de **son entreprise mère**, et non les siennes propres. Le pipeline
> s'exécuterait sans erreur tout en produisant des données fausses — le type
> de bug qui ne se détecte qu'à la lecture attentive d'un résultat.

La fonction est écrite une seule fois de façon générique : la question 5 (les
établissements) suit exactement la même structure ; seuls la collection source
et le nom de la clé enfant diffèrent.

In [9]:
def _children_lookup(*, source: str, child_key: str, alias: str) -> dict:
    """Associe à une entreprise ses entités filles, déjà enrichies de leurs propres détails.

    - `source`    : collection des entités filles (`kbo_branch` ou `kbo_establishment`)
    - `child_key` : clé propre de l'entité fille (`Id` ou `EstablishmentNumber`),
                    transmise à `_detail_lookups` dans le sous-pipeline
    - `alias`     : nom du tableau résultant dans le document entreprise
    """
    return {"$lookup": {
        "from": source,
        # (a) corrélation explicite entreprise -> fille
        "let": {"enterprise_number": "$EnterpriseNumber"},
        "pipeline": [
            {"$match": {"$expr": {"$eq": ["$EnterpriseNumber", "$$enterprise_number"]}}},
            # (b) chaque fille récupère ses propres détails via SA clé
            *_detail_lookups(child_key),
        ],
        "as": alias,
    }}


def _branch_lookup() -> dict:
    """Construit l'étape de lookup pour les succursales, reliées par `Id` (question 4)."""
    return _children_lookup(source="kbo_branch", child_key="Id", alias="branches")


import json
print(json.dumps(_branch_lookup(), indent=2))

{
  "$lookup": {
    "from": "kbo_branch",
    "let": {
      "enterprise_number": "$EnterpriseNumber"
    },
    "pipeline": [
      {
        "$match": {
          "$expr": {
            "$eq": [
              "$EnterpriseNumber",
              "$$enterprise_number"
            ]
          }
        }
      },
      {
        "$lookup": {
          "from": "kbo_denomination",
          "localField": "Id",
          "foreignField": "EntityNumber",
          "as": "denominations"
        }
      },
      {
        "$lookup": {
          "from": "kbo_address",
          "localField": "Id",
          "foreignField": "EntityNumber",
          "as": "addresses"
        }
      },
      {
        "$lookup": {
          "from": "kbo_contact",
          "localField": "Id",
          "foreignField": "EntityNumber",
          "as": "contacts"
        }
      },
      {
        "$lookup": {
          "from": "kbo_activity",
          "localField": "Id",
          "foreignField": "EntityNumber",


---

## 5) Rattacher les établissements

Même exercice que la question 4, mais appliqué aux établissements, qui sont les
unités opérationnelles d'une entreprise belge.

Comme pour les succursales, chaque établissement doit déjà intégrer ses propres dénominations,
adresses, contacts et activités (obtenus via la fonction de la question 3) avant
d'être associé à son entreprise.

L'énoncé dit « même exercice », et c'est littéralement exact : la structure est
identique, seuls deux paramètres varient. `_children_lookup` ayant été conçue
de manière générique à la question 4, la réponse tient en une seule ligne.

| | Question 4 | Question 5 |
|---|---|---|
| collection source | `kbo_branch` | `kbo_establishment` |
| clé propre de l'entité fille | `Id` | `EstablishmentNumber` |
| lien vers l'entreprise | `EnterpriseNumber` | `EnterpriseNumber` |
| alias produit | `branches` | `establishments` |

C'est également le lookup **le plus coûteux du pipeline** : 1,69 million
d'établissements, chacun déclenchant à son tour 4 jointures. Il représente
à lui seul l'essentiel du temps d'exécution de la question 6.

In [10]:
def _establishment_lookup() -> dict:
    """Construit l'étape de lookup pour les établissements, reliés par `EstablishmentNumber` (question 5)."""
    return _children_lookup(source="kbo_establishment",
                            child_key="EstablishmentNumber",
                            alias="establishments")


print(json.dumps(_establishment_lookup(), indent=2))

{
  "$lookup": {
    "from": "kbo_establishment",
    "let": {
      "enterprise_number": "$EnterpriseNumber"
    },
    "pipeline": [
      {
        "$match": {
          "$expr": {
            "$eq": [
              "$EnterpriseNumber",
              "$$enterprise_number"
            ]
          }
        }
      },
      {
        "$lookup": {
          "from": "kbo_denomination",
          "localField": "EstablishmentNumber",
          "foreignField": "EntityNumber",
          "as": "denominations"
        }
      },
      {
        "$lookup": {
          "from": "kbo_address",
          "localField": "EstablishmentNumber",
          "foreignField": "EntityNumber",
          "as": "addresses"
        }
      },
      {
        "$lookup": {
          "from": "kbo_contact",
          "localField": "EstablishmentNumber",
          "foreignField": "EntityNumber",
          "as": "contacts"
        }
      },
      {
        "$lookup": {
          "from": "kbo_activity",
          "loc

---

## 6) Assembler et exécuter le pipeline complet

Combinez les fonctions précédentes en un unique pipeline d'agrégation, exécuté
sur la collection des entreprises : d'abord les quatre informations complémentaires
de l'entreprise elle-même, puis ses établissements, puis ses succursales.

Le résultat de ce pipeline doit être écrit dans une nouvelle collection.

Exécutez le pipeline, puis affichez le document complet d'une entreprise disposant
d'au moins un établissement, et d'une entreprise disposant d'au moins une succursale.

### Assemblage

Le pipeline reflète fidèlement l'énoncé :

```python
[
    *_detail_lookups("EnterpriseNumber"),   # les 4 détails de l'entreprise
    _establishment_lookup(),                # puis ses établissements  (Q5)
    _branch_lookup(),                       # puis ses succursales     (Q4)
    {"$out": "entreprise"},                 # écriture du résultat
]
```

Deux paramètres d'exécution méritent une explication :

- **`$out`** écrit le résultat dans une nouvelle collection en remplaçant
  atomiquement la collection cible. Le pipeline est donc rejouable sans risque
  de doublons, et les lecteurs ne voient jamais d'état intermédiaire.
  (`$merge` conviendrait pour un rafraîchissement incrémental ; ici on reconstruit
  un snapshot complet, `$out` est plus simple et plus rapide.)
- **`allowDiskUse=True`** permet aux étapes d'écrire sur disque au-delà des
  100 Mo de RAM alloués par défaut. Sur ces volumes, c'est incontournable.

> **Note de volumétrie** : le document le plus lourd correspond à l'entreprise
> `0214.596.464` et ses 1 058 établissements, soit environ 3 Mo — bien en deçà
> de la limite BSON de 16 Mo par document. Cette vérification n'est pas optionnelle
> sur un modèle imbriqué : c'est elle qui détermine si une dénormalisation est
> réellement envisageable.

L'exécution nécessite plusieurs dizaines de minutes : 1,95 M d'entreprises et
1,69 M d'établissements génèrent au total près de 15 millions de recherches indexées.

In [11]:
def build_pipeline(target: str = "entreprise") -> list[dict]:
    """Construit et retourne le pipeline d'agrégation complet de la couche Bronze."""
    return [
        *_detail_lookups("EnterpriseNumber"),   # détails de l'entreprise (Q3)
        _establishment_lookup(),                # établissements avec leurs détails (Q5)
        _branch_lookup(),                       # succursales avec leurs détails (Q4)
        {"$out": target},                       # écriture dans la collection cible (Q6)
    ]


pipeline = build_pipeline()
print(f"{len(pipeline)} etapes :",
      [next(iter(stage)) for stage in pipeline])

7 etapes : ['$lookup', '$lookup', '$lookup', '$lookup', '$lookup', '$lookup', '$out']


In [12]:
started = time.perf_counter()
db.kbo_enterprise.aggregate(pipeline, allowDiskUse=True)
elapsed = time.perf_counter() - started

print(f"collection `entreprise` construite en {elapsed / 60:.1f} min")
print(f"{db.entreprise.count_documents({}):,} documents")

collection `entreprise` construite en 22.7 min


1,955,776 documents


### Une entreprise avec au moins un établissement

`0403.449.823` : deux établissements, chacun portant ses propres dénominations,
adresses et activités. Le tableau `branches` est vide — il s'agit d'une entreprise
belge sans présence étrangère.

In [13]:
import json

def show(document: dict) -> None:
    print(json.dumps(document, indent=2, ensure_ascii=False, default=str))

show(db.entreprise.find_one({"_id": "0403.449.823"}))

{
  "_id": "0403.449.823",
  "EnterpriseNumber": "0403.449.823",
  "Status": "AC",
  "JuridicalSituation": "000",
  "TypeOfEnterprise": "2",
  "JuridicalForm": "014",
  "JuridicalFormCAC": "",
  "StartDate": "01-01-1968",
  "denominations": [
    {
      "_id": "6a69f7ecbbadb5ac70a6a5a2",
      "EntityNumber": "0403.449.823",
      "Language": "2",
      "TypeOfDenomination": "001",
      "Denomination": "Thornton"
    }
  ],
  "addresses": [
    {
      "_id": "6a69f823bbadb5ac70d9c87f",
      "EntityNumber": "0403.449.823",
      "TypeOfAddress": "REGO",
      "CountryNL": "",
      "CountryFR": "",
      "Zipcode": "2030",
      "MunicipalityNL": "Antwerpen",
      "MunicipalityFR": "Antwerpen",
      "StreetNL": "Treurenborg",
      "StreetFR": "Treurenborg",
      "HouseNumber": "9",
      "Box": "",
      "ExtraAddressInfo": "",
      "DateStrikingOff": ""
    }
  ],
  "contacts": [],
  "activities": [
    {
      "_id": "6a69f8bfbbadb5ac7010de2c",
      "EntityNumber": "0403.449

### Une entreprise avec au moins une succursale

`0257.883.408` : une association turque implantée en Belgique. Son adresse
principale est en Turquie ; elle dispose d'une succursale (`branches`) en plus
d'un établissement local.

In [14]:
show(db.entreprise.find_one({"_id": "0257.883.408"}))

{
  "_id": "0257.883.408",
  "EnterpriseNumber": "0257.883.408",
  "Status": "AC",
  "JuridicalSituation": "000",
  "TypeOfEnterprise": "2",
  "JuridicalForm": "030",
  "JuridicalFormCAC": "",
  "StartDate": "01-09-1995",
  "denominations": [
    {
      "_id": "6a69f7ecbbadb5ac70a67d13",
      "EntityNumber": "0257.883.408",
      "Language": "1",
      "TypeOfDenomination": "001",
      "Denomination": "ASSOCIATION TURQUE DES EXPORTATEURS DE TEXTILE ET D'HABILLEMENT D'ISTANBUL - ITKIB"
    }
  ],
  "addresses": [
    {
      "_id": "6a69f823bbadb5ac70d9a6af",
      "EntityNumber": "0257.883.408",
      "TypeOfAddress": "REGO",
      "CountryNL": "Turkije",
      "CountryFR": "Turquie",
      "Zipcode": "34196",
      "MunicipalityNL": "yenibosna - Istamboul",
      "MunicipalityFR": "yenibosna - Istamboul",
      "StreetNL": "itkib bis ticaret komplexi b/blok coban cesme mekvil sanayi/caddesi",
      "StreetFR": "itkib bis ticaret komplexi b/blok coban cesme mekvil sanayi/caddesi",
 

### Contrôle final

On s'assure que le pipeline n'a **écarté aucune entreprise** (le nombre de documents
produits doit être identique au nombre d'entreprises en entrée) et que les entités
filles ont correctement été rattachées.

In [15]:
enterprises = db.kbo_enterprise.count_documents({})
produced = db.entreprise.count_documents({})

print(f"entreprises en entree : {enterprises:>10,}")
print(f"documents produits    : {produced:>10,}   {'OK' if produced == enterprises else 'ECART'}")
print(f"avec >= 1 etablissement: {db.entreprise.count_documents({'establishments.0': {'$exists': True}}):>10,}")
print(f"avec >= 1 succursale   : {db.entreprise.count_documents({'branches.0': {'$exists': True}}):>10,}")
print(f"avec >= 1 activite     : {db.entreprise.count_documents({'activities.0': {'$exists': True}}):>10,}")

stats = db.command("collStats", "entreprise", scale=1024 * 1024)
print(f"\ntaille `entreprise`   : {stats['size']:,.0f} Mo "
      f"({stats['storageSize']:,.0f} Mo sur disque)")
print(f"document moyen        : {stats['avgObjSize'] / 1024:,.1f} Ko")

entreprises en entree :  1,955,776
documents produits    :  1,955,776   OK


avec >= 1 etablissement:  1,527,664


avec >= 1 succursale   :      7,313


avec >= 1 activite     :  1,263,997

taille `entreprise`   : 7,160 Mo (780 Mo sur disque)
document moyen        : 3.7 Ko


---

## Bilan

La couche Bronze est opérationnelle : **un document autonome par entreprise**,
produit en une seule passe batch, là où il fallait auparavant huit lectures dispersées.

Points clés à retenir :

| Choix | Justification |
|---|---|
| Chargement par lots en streaming | empreinte mémoire constante sur des fichiers de plusieurs Go |
| `_id = EnterpriseNumber` | clé naturelle : index offert + chargement idempotent |
| Index créés avant les `$lookup` | sans eux, le pipeline ne converge pas |
| Une fonction `_detail_lookups`, trois appels | les 4 détails se rattachent de la même façon quel que soit le niveau |
| `let` + `$expr` pour les entités filles | deux clés distinctes gérées dans une seule étape |
| `$out` | remplacement atomique de la collection cible, pipeline rejouable sans risque |

Ce que la couche Bronze **n'a délibérément pas fait** : traduire `Status="AC"`,
trancher entre `MunicipalityNL` et `MunicipalityFR`, ou dédoublonner les activités
qui réapparaissent sous plusieurs versions NACE. Ce travail est confié au second
notebook, la couche **Silver**.